# Check the checkout test's split

*Data Analysis in Python · Topic 9.2*

Before the product manager announces B's lift, check that the test split its users the way it was designed: half and half.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

1. Run the parameters cell, the cell that reads the test's two files, and the cell that builds the `users` table: one row per user in the test, with `converted` and `revenue` (topic 9.1.'s steps).
2. In the first cell marked ✏️, write the function `srm_p_value(counts)`: it takes the number of users in each variant, such as `[5000, 5200]` or a Series, and returns the p-value of a chi-square test that they were split evenly. Give it a docstring.
3. In the second cell marked ✏️, create:
   - `variant_counts`: the number of users in each variant, as a Series
   - `share_b`: B's share of the users, as a fraction
   - `srm_p`: `srm_p_value()`'s p-value for `variant_counts`
   - `srm_detected`: `True` if `srm_p` is below 0.001, the threshold for an SRM check
4. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output.

**Expected output**: `srm_p_value()` returns 1.0 for `[5000, 5000]` and about 0.048 for `[5000, 5200]`. `variant_counts` has A 41,706 and B 38,348, `share_b` is about `0.4790`, `srm_p` about `1.7e-32`, and `srm_detected` is `True`.

## Parameters

In [ ]:
import pandas as pd
from scipy import stats

DATA_URL = "https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/"

In [ ]:
# The test's two files, with the user IDs as text. Run it, and leave it as it is.
assignments = pd.read_csv(DATA_URL + "ab_test_assignments.csv", dtype={"user_pseudo_id": str}, parse_dates=["assigned_at"])
conversions = pd.read_csv(DATA_URL + "ab_test_conversions.csv", dtype={"user_pseudo_id": str}, parse_dates=["converted_at"])
assignments.shape, conversions.shape

In [ ]:
# One row per user in the test, with converted and revenue (topic 9.1). Run it, and leave it as it is.
users = assignments.merge(conversions[["user_pseudo_id", "revenue"]], on="user_pseudo_id", how="left", validate="one_to_one")
users["converted"] = users["revenue"].notna()
users["revenue"] = users["revenue"].fillna(0)
users.shape

## The check

In [ ]:
# ✏️ Write srm_p_value(counts)


## The split

In [ ]:
# ✏️ Create variant_counts, share_b, srm_p and srm_detected


In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables, calls your function and reads the cells that have run. It doesn't change anything.
def check_your_work():
    import sys
    import numpy as np
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            results.append((True, "Every cell in this session ran once, with no errors."))

    def number(value):
        return pd.api.types.is_number(value) and not isinstance(value, (pd.DataFrame, pd.Series))

    srm_p_value = found.get("srm_p_value")
    if not callable(srm_p_value):
        results.append((False, "srm_p_value should be a function: def srm_p_value(counts): ... return stats.chisquare(counts).pvalue."))
    else:
        try:
            even = srm_p_value(pd.Series({"A": 5000, "B": 5000}))
            uneven = srm_p_value(pd.Series({"A": 5000, "B": 5200}))
        except Exception as error:
            even = uneven = None
            results.append((False, f"srm_p_value(pd.Series({{\"A\": 5000, \"B\": 5200}})) stopped with {type(error).__name__}: {error}. It should take one argument, the counts, as a list or a Series."))
        if even is not None:
            if not number(even) or not number(uneven):
                results.append((False, f"srm_p_value() should return one number, the p-value: stats.chisquare(counts).pvalue. For [5000, 5200] it returns {uneven!r}."))
            elif abs(uneven - 3.9216) < 0.001:
                results.append((False, "srm_p_value([5000, 5200]) returns 3.92: that's the chi-square statistic. Return the p-value: stats.chisquare(counts).pvalue."))
            elif abs(even - 1) > 0.000001 or not (abs(uneven - 0.04767) < 0.0001 or abs(uneven - 0.04879) < 0.0001):
                results.append((False, f"srm_p_value() should return about 1.0 for [5000, 5000] and 0.048 for [5000, 5200]. It returns {even:.4f} and {uneven:.4f}."))
            elif not (srm_p_value.__doc__ or "").strip():
                results.append((False, "srm_p_value() has no docstring. Say what it returns, in a line under the def."))
            else:
                results.append((True, f"srm_p_value() returns {even:.1f} for an even split of 10,000 users and {uneven:.3f} for 5,000 against 5,200."))

    counts = found.get("variant_counts")
    share_b, srm_p, detected = found.get("share_b"), found.get("srm_p"), found.get("srm_detected")
    if not isinstance(counts, pd.Series) or not {"A", "B"} <= set(counts.index):
        results.append((False, "variant_counts should be a Series with the number of users in A and in B: users[\"variant\"].value_counts()."))
    elif counts["A"] == 1907 or counts["B"] == 1944:
        results.append((False, "variant_counts has the conversions, 1,907 and 1,944. The test splits users, not conversions: count the users table's rows by variant."))
    elif counts["A"] != 41706 or counts["B"] != 38348:
        results.append((False, f"variant_counts should be A 41,706 and B 38,348. It is {counts['A']:,} and {counts['B']:,}."))
    elif not number(share_b):
        results.append((False, f"share_b should be a number: variant_counts[\"B\"] / variant_counts.sum(). It is {share_b!r}."))
    elif abs(share_b - 0.520973) < 0.000001:
        results.append((False, "share_b is 0.5210: that's A's share. Divide B's users by all the users."))
    elif abs(share_b - 47.9027) < 0.001:
        results.append((False, "share_b is in percent, 47.90. Keep it as a fraction, 0.4790."))
    elif abs(share_b - 0.479027) > 0.000001:
        results.append((False, f"share_b should be B's users divided by all users, about 0.4790. It is {share_b:.4f}."))
    elif not number(srm_p):
        results.append((False, f"srm_p should be a number: srm_p_value(variant_counts). It is {srm_p!r}."))
    elif abs(srm_p - 0.5510) < 0.0001:
        results.append((False, "srm_p is 0.5510: that's the test on the conversions. The SRM check is about the users assigned to each variant."))
    elif not 0 <= srm_p < 1e-25:
        results.append((False, f"srm_p should be the chi-square p-value for 41,706 and 38,348 users: about 1.7e-32. It is {srm_p:.3g}."))
    elif not isinstance(detected, (bool, np.bool_)):
        results.append((False, f"srm_detected should be True or False: srm_p < 0.001. It is {detected!r}."))
    elif not detected:
        results.append((False, "srm_detected is False, but srm_p is far below 0.001: srm_detected = srm_p < 0.001."))
    else:
        results.append((True, f"A sample ratio mismatch: B has {share_b:.1%} of the users, and a 50/50 split would give one this uneven with a chance of {srm_p:.1e}. Don't report the lift until you know why."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run.")


check_your_work()

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
def srm_p_value(counts):
    """Return the p-value of a chi-square test that the counts were split evenly between the variants."""
    return stats.chisquare(counts).pvalue


srm_p_value([5000, 5000]), srm_p_value([5000, 5200])
```

```python
variant_counts = users["variant"].value_counts()
share_b = variant_counts["B"] / variant_counts.sum()
srm_p = srm_p_value(variant_counts)
srm_detected = srm_p < 0.001
variant_counts, share_b, srm_p, srm_detected
```

</details>